# 1. Building a preprocessing pipeline

## 1.1 Tokenization

Splits a sentence/ paragraph into smaller units /words/ phrases) and serve as the bulding blocks for text analysis

In [1]:
from torchtext.data.utils import get_tokenizer

In [2]:
tokenizer = get_tokenizer("basic_english")
text = "I am reading a book now. I love to read books!"
tokens = tokenizer(text)
print(tokens)

['i', 'am', 'reading', 'a', 'book', 'now', '.', 'i', 'love', 'to', 'read', 'books', '!']


## 1.2 Removing Stop Words

 These are common words like 'a', 'the' and 'and' that often do not add significant meaning. Removing them reduces the dataset size without losing valuable information.

In [3]:
import nltk
from nltk.corpus import stopwords

In [4]:
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))
tokens = [token for token in tokens if token.lower() not in stop_words]
print(tokens)

['reading', 'book', '.', 'love', 'read', 'books', '!']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\flash\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 1.3 Stemming

Stemming reduces words to their root-form. Example: "running", "runs" and "ran" all become "run"

In [5]:
from nltk.stem import PorterStemmer

In [6]:
stemmer = PorterStemmer()
stemmed_tokens = [stemmer.stem(token) for token in tokens]
print(stemmed_tokens)

['read', 'book', '.', 'love', 'read', 'book', '!']


## 1.4 Removing rare words

Infrequent words often add littel value, especially in large datasets. By removing them, we ensure that the model focuses on commonly occuring patterns

In [7]:
from nltk.probability import FreqDist

In [8]:
freq_dist = FreqDist(stemmed_tokens)
threshold = 2
common_tokens = [token for token in stemmed_tokens if freq_dist[token] >= threshold]
print(common_tokens)

['read', 'book', 'read', 'book']


## 2. Bring together the completed pipeline

In [9]:
from torchtext.data.utils import get_tokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.probability import FreqDist
import nltk

In [10]:
# Download required resources
nltk.download('stopwords')
def preprocess_text(text, frequency_threshold=2):
    # Initialize tools
    tokenizer = get_tokenizer("basic_english")
    stop_words = set(stopwords.words('english'))
    stemmer = PorterStemmer()
    # Tokenize
    tokens = tokenizer(text)
    # Remove stop words
    tokens = [token for token in tokens if token.lower() not in stop_words]
    # Apply stemming
    tokens = [stemmer.stem(token) for token in tokens]
    # Remove rare words
    freq_dist = FreqDist(tokens)
    tokens = [token for token in tokens if freq_dist[token] >= frequency_threshold]
    return tokens
# Example usage
text = "I am reading a book now. I love to read books!"
processed_tokens = preprocess_text(text)
print(processed_tokens)

['read', 'book', 'read', 'book']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\flash\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# 2. Text Classification

1. Binary Classification: Is an email "Spam" or "Not-Spam" ?
2. Multi-Class Classification: Is a news article about "Politics", "Sports" or "Technology" ?
3. Multi-Label Classification: Assign multiple genres to a book, like "Fantasy", "Adventure" and "Action"

## 2.0 Data Preparation

In [11]:
# example dataset
corpus = [
    ("The stock market is performing well today.", "Finance"),
    ("The soccer team wont their last match.", "Sports"),
    ("The new technology is groundbracking.", "Technology")
]
sentences, labels = zip(*corpus)

### 2.0.1 Encoding Text: Bag-of-Words

Popular technique for text encoding. converts sentences into numerical representations based on word frequency

In [12]:
from sklearn.feature_extraction.text import CountVectorizer

In [13]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(sentences)
print("Feature Names:", vectorizer.get_feature_names_out())
print("Encoded Sentences:\n", X.toarray())

Feature Names: ['groundbracking' 'is' 'last' 'market' 'match' 'new' 'performing' 'soccer'
 'stock' 'team' 'technology' 'the' 'their' 'today' 'well' 'wont']
Encoded Sentences:
 [[0 1 0 1 0 0 1 0 1 0 0 1 0 1 1 0]
 [0 0 1 0 1 0 0 1 0 1 0 1 1 0 0 1]
 [1 1 0 0 0 1 0 0 0 0 1 1 0 0 0 0]]


## 2.1 Simple Text Classification Model

### 2.1.1 Define the Dataset

In [14]:
import torch
from torch.utils.data import Dataset

In [15]:
class TextClassificationDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx].toarray(), dtype = torch.float32).squeeze(0), self.y[idx]

label_mapping = {"Finance": 0, "Sports": 1, "Technology": 2}
encoded_labels = [label_mapping[label] for label in labels]
dataset = TextClassificationDataset(X, encoded_labels)

### 2.1.2 Define the Model

In [16]:
import torch.nn as nn

In [17]:
class TextClassifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(TextClassifier, self).__init__()
        self.fc = nn.Linear(input_dim, output_dim)
    def forward(self, x):
        return self.fc(x)

input_dim = X.shape[1]
output_dim = len(label_mapping)
model = TextClassifier(input_dim, output_dim)

### 2.1.3 Training the Model

In [18]:
from torch.utils.data import DataLoader
import torch.optim as optim
from tqdm import tqdm

In [19]:
dataloader = DataLoader(dataset, batch_size = 2, shuffle = True)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = 0.01)

for epoch in tqdm(range(50)):
    for inputs, labels in dataloader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

100%|█████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 318.64it/s]

Epoch 1, Loss: 1.3384
Epoch 2, Loss: 1.5102
Epoch 3, Loss: 1.2563
Epoch 4, Loss: 1.4123
Epoch 5, Loss: 1.3477
Epoch 6, Loss: 1.2861
Epoch 7, Loss: 0.5916
Epoch 8, Loss: 1.1223
Epoch 9, Loss: 1.0740
Epoch 10, Loss: 1.1587
Epoch 11, Loss: 1.1064
Epoch 12, Loss: 0.5853
Epoch 13, Loss: 0.5739
Epoch 14, Loss: 0.5629
Epoch 15, Loss: 1.0008
Epoch 16, Loss: 0.5522
Epoch 17, Loss: 0.5417
Epoch 18, Loss: 0.9233
Epoch 19, Loss: 0.5314
Epoch 20, Loss: 0.5215
Epoch 21, Loss: 0.8534
Epoch 22, Loss: 0.8577
Epoch 23, Loss: 0.5127
Epoch 24, Loss: 0.8118
Epoch 25, Loss: 0.5039
Epoch 26, Loss: 0.7643
Epoch 27, Loss: 0.4938
Epoch 28, Loss: 0.7215
Epoch 29, Loss: 0.7371
Epoch 30, Loss: 0.6820
Epoch 31, Loss: 0.4833
Epoch 32, Loss: 0.6458
Epoch 33, Loss: 0.6803
Epoch 34, Loss: 0.4733
Epoch 35, Loss: 0.6036
Epoch 36, Loss: 0.6386
Epoch 37, Loss: 0.5733
Epoch 38, Loss: 0.4616
Epoch 39, Loss: 0.5456
Epoch 40, Loss: 0.5267
Epoch 41, Loss: 0.5853
Epoch 42, Loss: 0.5651
Epoch 43, Loss: 0.5460
Epoch 44, Loss: 0.44

### 2.1.4 Evaluate the Model

In [20]:
new_sentence = "The new software release is amazing."
new_X = vectorizer.transform([new_sentence])

with torch.no_grad():
    prediction = model(torch.tensor(new_X.toarray(), dtype = torch.float32))
    predicted_label = torch.argmax(prediction).item()
    print("Predicted Label:", list(label_mapping.keys())[predicted_label])

Predicted Label: Technology


# 3 Text Generation

## 3.1 Explanation

A fascinationg application of natual language processing (NLP). Allows to create chatbots, summarize content, or text prediction

## 3.2 Character Lebel RNN for TExtGeneration

### 3.2.1 Preparing the Dataset

In [21]:
import torch
import torch.nn as nn

In [22]:
data = "Hello, how are you?"
chars = list(set(data)) # get the indiviidual, unqiue symbols
## Creating mappings
char_to_idx = {char: i for i, char in enumerate(chars)} # assign each symbol a number/ index
idx_to_char = {i: char for char, i in char_to_idx.items()}
## Prepare input and target sequences
inputs = [char_to_idx[ch] for ch in data[:-1]] # "Translate" the sentence into the index-numbers. Skip the last symbol
targets = [char_to_idx[ch] for ch in data[1:]] # "Translate" the sentence into the index-numbers. Skip the first symbol
## Convert to tensors
inputs = torch.tensor(inputs, dtype = torch.long).unsqueeze(0)
targets = torch.tensor(targets, dtype = torch.long).unsqueeze(0)
print(inputs)
print(targets)

tensor([[10,  0, 11, 11,  8,  3,  6,  1,  8,  4,  6,  2,  7,  0,  6,  5,  8,  9]])
tensor([[ 0, 11, 11,  8,  3,  6,  1,  8,  4,  6,  2,  7,  0,  6,  5,  8,  9, 12]])


### 3.2.2 Defining the RNN Model

In [23]:
class RNNModel(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super(RNNModel, self).__init__()
        self.hidden_size = hidden_size
        # often used to store word embeddings and retrieve them using indices
        # num_embeddings = vocab_size, embedding_dim = hidden_size
        # input is list of indices
        # output is corresponding word embedding
        ### We transform our "symbol" tensor with possible indices of 0 to 12 into an embedding vecotor of size 128
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        # input_size = hidden_size, hidden_size = hidden_size
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first = True)
        self.fc = nn.Linear(hidden_size, vocab_size) # we squeeze it back from the embedded veccotr o the list of 13 charaters (12 + the 1 predicted)

    def forward(self, x, hidden):
        x = self.embedding(x)
        out_rnn, hidden = self.rnn(x, hidden)
        out = self.fc(out_rnn)
        return out, out_rnn
        
    def init_hidden(self):
        return torch.zeros(1, 1, self.hidden_size)

vocab_size = len(chars)
hidden_size = 128
model = RNNModel(vocab_size, hidden_size)

In [24]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    hidden = model.init_hidden()
    # Zero gradients
    optimizer.zero_grad()
    # Forward pass
    outputs, hidden = model(inputs, hidden)
    # The loss function takes over the application of a softmax.
    # outputs contains 18 vecotrs of length 13. Eeach of these 18 vecotrs has a score for each of the 13 characters. 
    loss = criterion(outputs.view(-1, vocab_size), targets.view(-1))
    # Backward pass
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [10/100], Loss: 2.2822
Epoch [20/100], Loss: 1.9582
Epoch [30/100], Loss: 1.6759
Epoch [40/100], Loss: 1.4341
Epoch [50/100], Loss: 1.2307
Epoch [60/100], Loss: 1.0623
Epoch [70/100], Loss: 0.9241
Epoch [80/100], Loss: 0.8113
Epoch [90/100], Loss: 0.7190
Epoch [100/100], Loss: 0.6430


### 3.2.3 Generating Text

In [31]:
def generate_text(model, start_char, length):
    model.eval()
    generated_text = start_char
    hidden = model.init_hidden()

    # "transform" character into predetermined index
    input_char = torch.tensor([char_to_idx[start_char]], dtype = torch.long).unsqueeze(0)
    for _ in range(length):
        output, hidden = model(input_char, hidden)
        predicted_idx = torch.argmax(output, dim=2).item() # get the index with the highest score
        next_char = idx_to_char[predicted_idx] # get the character assigned to the hightest scoring index
        generated_text += next_char # append it to the output-sentence
        input_char = torch.tensor([[predicted_idx]], dtype = torch.long)
    return generated_text

seed_char = "H"
print(generate_text(model, seed_char, 50))

Hello, how are you?, how are you?, how are you?, ho


## 3.3 Transformers

Backbone of modern NLP models like BERT or GPT. Designed to understand relationships between words. Unlike RNNs, process sequences in parallel, making them faster and more effective for tasks requiring context

### 3.3.1 Key Components

1. Encoder
2. Decoder
3. Positional Encoding
4. Attention (Self or Multi-Head)
5. Feedforward Networks

In [32]:
import torch
import torch.nn as nn

In [35]:
class TransformerEncoder(nn.Module):
    def __init__(self, embed_size, heads, num_layers, dropout):
        super(TransformerEncoder, self).__init__()
        self.encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(d_model = embed_size, nhead = heads),
                                             num_layers = num_layers
                                             )
        self.fc = nn.Linear(embed_size, 2) # Binary Classification
    def forward(self, x):
        x = self.encoder(x)
        x = x.mean(dim = 1) # Global average pooling
        return self.fc(x)
model = TransformerEncoder(embed_size = 512, heads = 8, num_layers = 3, dropout = 0.5)

C:\Users\flash\anaconda3\envs\ML_Pytorch\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


### 3.3.2 Attention

Enables the model to dicus on the most relevant parts of a sequence. They assign different weights to words based on their significance in a given context </br>
1. Self-Attention: Relates words within the same sequence
2. Multi-Head Attention: Examines multiple aspects of relationships simultaneously

In [41]:
class SelfAttention(nn.Module):
    def __init__(self, embed_size):
        super(SelfAttention, self).__init__()
        self.query = nn.Linear(embed_size, embed_size)
        self.key = nn.Linear(embed_size, embed_size)
        self.value = nn.Linear(embed_size, embed_size)
    def forward(self, x):
        queries = self.query(x)
        keys = self.key(x)
        values = self.value(x)
        attention_scores = torch.matmul(queries, keys.transpose(-1, -2)) / (x.size(-1)**0.5)
        attention_weights = torch.nn.functional.softmax(attention_scores, dim = -1)
        context = torch.matmul(attention_weights, values)
        return context

# Example usage
attention = SelfAttention(embed_size = 512)
output = attention(torch.rand(32, 10, 512)) # Batch size of 32, sequence length of 10

### 3.3.3 Defending against adversarial attacks

These subtly modify input data to deceive NLP models. Like "good" to "g0od" could trick a model into misclassifying sentiment </br>
1. Fast Gradient Sign Method (FGSM): makes small, calculated changes to input data
2. Projected Gradient Descent (PGD) Iteratively applies FGSM for more robust attacks
3. Carlini & Wagner (C&W): Optimizes the loss funciton to create undetectable yet harmful modifications </br>
One effective defense is adversarial training, where the model is trained on both clean and adversarially perturbed data

In [44]:
def fgsm_attack(model, data, target, epsilon):
    data.requires_grad = True
    output = model(data)
    loss = nn.CrossEntropyLoss()(output, target)
    model.zero_grad()
    loss.backward()
    perturbation = epsilon * data.grad.sign()
    adversarial_data = data + perturbation
    return adversarial_data

# Example usage
adversarial_data = fgsm_attack(model, torch.rand(1, 10, 512), torch.tensor([1]), epsilon = 0.1)

### 3.3.4 Putting it all together

In [47]:
class SentimentTransformer(nn.Module):
    def __init__(self, vocab_size, embed_size, heads, num_layers, num_classes):
        super(SentimentTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model = embed_size, nhead = heads), 
            num_layers = num_layers
        )
        self.attention = SelfAttention(embed_size)
        self.fc = nn.Linear(embed_size, num_classes)
    def forward(self, x):
        x = self.embedding(x)
        x = self.encoder(x)
        x = self.attention(x).mean(dim = 1)
        return self.fc(x)
        
model = SentimentTransformer(vocab_size = 10000, embed_size = 512, heads = 8, num_layers = 3, num_classes = 2)
